In [175]:
import pandas as pd
import unicodedata
import re
import numpy as np

# 0 Funcoes

## Funções Bryan

In [176]:
# Como estamos importando de um excel, vamos ter que ajustar o nome das colunas
def normalizar_colunas(df):
    def remover_acentos(texto):
        return ''.join(
            c for c in unicodedata.normalize('NFKD', texto)
            if not unicodedata.combining(c)
        )

    df = df.copy()
    df.columns = [
        remover_acentos(col)
            .lower()
            .replace(' ', '_')
        for col in df.columns
    ]

    return df


In [177]:
# Como vamos juntar alguns dataframes, é melhor que estejam com os mesmos nomes algumas colunas
def renomear_colunas(df, mapa_colunas):
    """
    Parâmetros:
    df (pd.DataFrame): DataFrame original
    mapa_colunas (dict): {'nome_antigo': 'nome_novo'}

    Retorna:
    pd.DataFrame: DataFrame com colunas renomeadas
    """
    df = df.copy()

    # Aplica somente às colunas que existem no DataFrame
    mapa_valido = {
        col_antiga: col_nova
        for col_antiga, col_nova in mapa_colunas.items()
        if col_antiga in df.columns
    }

    df.rename(columns=mapa_valido, inplace=True)
    return df


In [178]:
"""
Como os dataframes tem colunas diferentes, na hora de juntar essa função irá ajudar.
Iremos adicionar colunas nos outros dataframes para que a junção possa ocorrer (as colunas terão dasdos vazios)
"""
def adaptar_dataframe(df, colunas_base, origem, lista_ids):
    df=df.copy()

     # filtra apenas os IDs desejados
    df = df[df['ra'].isin(lista_ids)]

    # adiciona colunas que faltam
    for col in colunas_base:
        if col not in df.columns:
            df[col] = pd.NA

    # mantém apenas as colunas do principal
    df = df[colunas_base]

    # cria coluna de origem
    df['ano_dataframe'] = origem

    return df


In [179]:
# Função correção de tipo de colunas
def corrigir_dados(tipo, dataframe, colunas):
    """
    Corrige o tipo de dados de colunas de um DataFrame.

    Parâmetros:
    tipo (str): tipo alvo ('int', 'float', 'str', 'data')
    dataframe (pd.DataFrame): DataFrame original
    colunas (list ou str): coluna ou lista de colunas

    Retorna:
    pd.DataFrame: DataFrame com colunas corrigidas
    """
    df = dataframe.copy()

    if isinstance(colunas, str):
        colunas = [colunas]

    match tipo.lower():
        case 'int':
            for col in colunas:
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

        case 'float':
            for col in colunas:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        case 'str':
            for col in colunas:
                df[col] = df[col].astype(str).str.strip()

        case 'data' | 'datetime':
            for col in colunas:
                df[col] = pd.to_datetime(df[col], errors='coerce')

        case _:
            raise ValueError(
                f"Tipo '{tipo}' não suportado. "
                "Use: int, float, str, data"
            )

    return df

In [180]:
def colunas_totalmente_vazias(df):
    """
    Identifica colunas que possuem apenas valores vazios e deleta

    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada
    remover (bool): se True, remove as colunas vazias

    Retorna:
    pd.DataFrame:
        - DataFrame sem colunas vazias (remover=True)
    """
    df_tmp = df.copy()

    # considera strings vazias como NaN
    df_tmp = df_tmp.replace(r'^\s*$', pd.NA, regex=True)

    colunas_vazias = [
        col for col in df_tmp.columns
        if df_tmp[col].isna().all()
    ]

    return df_tmp.drop(columns=colunas_vazias)


In [181]:
def arredondar_floats(df, casas=2):
    """
    Arredonda todas as colunas float de um DataFrame.

    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada
    casas (int): número de casas decimais (padrão = 2)

    Retorna:
    pd.DataFrame: DataFrame com floats arredondados
    """
    df = df.copy()

    colunas_float = df.select_dtypes(include=['float', 'float64', 'float32']).columns

    df[colunas_float] = df[colunas_float].round(casas)

    return df

## Funções Vitor

In [182]:

def normalizar_fase(valor):
    """
    Converte valores como '1A', '2C', '8F' ou 9 em 'FASE X'.
    Mantém valores sem número (ex: 'ALFA').
    """
    valor_str = str(valor)
    numeros = "".join(filter(str.isdigit, valor_str))
    
    if numeros:
        return f"FASE {numeros}"
    else:
        return valor

In [183]:
def remover_texto_parenteses(valor):
    """
    Remove qualquer texto entre parênteses e retorna o texto em MAIÚSCULO.
    Ex: 'Fase 1 (3° e 4° ano)' , 'Fase 2 (5° e 6° ano)', 'Fase 3 (7° e 8° ano)', 'Fase 4 (9° ano)',
       'Fase 6 (2° EM)', 'Fase 5 (1° EM)', 'Fase 7 (3° EM)',
       'Fase 8 (Universitários)' -> 'FASE 1', 'FASE 2', 'FASE 3', etc
    """
    if pd.isna(valor):
        return valor
    
    texto_limpo = re.sub(r"\s*\(.*?\)", "", str(valor)).strip()
    return texto_limpo.upper()

## Funções Luis

In [184]:
def genero_norm(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    if s in ["menina","feminino","f"]: return "Feminino"
    if s in ["menino","masculino","m"]: return "Masculino"
    return str(v).strip()

# 1 Importando os dados

In [185]:
arquivo = r"https://raw.githubusercontent.com/vbomura/tc5/0afbe2d22ee4658c81a7470b2360a2aef3cd032e/Base_Passos_Magicos/BASE%20DE%20DADOS%20PEDE%202024%20-%20DATATHON.xlsx"

# Pegando o dado de cada aba
base_2022 = pd.read_excel(arquivo, sheet_name="PEDE2022")
base_2023 = pd.read_excel(arquivo, sheet_name="PEDE2023")
base_2024 = pd.read_excel(arquivo, sheet_name="PEDE2024")

In [186]:
base_2022.columns

Index(['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero',
       'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21',
       'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1',
       'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3',
       'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA',
       'Matem', 'Portug', 'Inglês', 'Indicado', 'Atingiu PV', 'IPV', 'IAN',
       'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV'],
      dtype='object')

In [187]:
#Ajustando nomes identicos entre as baeses
base_2022 = base_2022.rename(columns={
    'Nome': 'Nome Anonimizado',
    'Idade 22': 'Idade',
    'Matem': 'Mat',
    'Portug': 'Por',
    'Inglês': 'Ing',
    'Defas': 'Defasagem',
    'Pedra 22': 'Pedra',
    'INDE 22': 'INDE'
})


#Ajustando nomes identicos entre as baeses
base_2023 = base_2023.rename(columns={
    'Pedra 2023': 'Pedra',
    'INDE 2023': 'INDE'
})

#Ajustando nomes identicos entre as baeses
base_2024 = base_2024.rename(columns={
    'Pedra 2024': 'Pedra',
    'INDE 2024': 'INDE'
})

In [188]:
#Criando coluna para ANO
base_2022['Ano_Aba'] = 2022
base_2022['Ano_Aba'] = base_2022['Ano_Aba'].astype(int)

base_2023['Ano_Aba'] = 2023
base_2023['Ano_Aba'] = base_2023['Ano_Aba'].astype(int)

base_2024['Ano_Aba'] = 2024
base_2024['Ano_Aba'] = base_2024['Ano_Aba'].astype(int)

In [189]:
#remover colunas dos dataframes 2022 
colunas_remover2022 = ['Ano nasc','Cg','Cf','Ct','Pedra 20','Pedra 21'
                   ,'Avaliador1','Rec Av1','Avaliador2','Rec Av2','Avaliador3','Rec Av3','Avaliador4'
                   ,'Rec Av4','Rec Psicologia','Indicado','Atingiu PV','Destaque IEG','Destaque IDA'
                   ,'Destaque IPV']

base_2022.drop(columns=colunas_remover2022, inplace=True)

In [190]:
#remover colunas dos dataframes 2023 
colunas_remover2023 = ['Data de Nasc','Pedra 20', 'Pedra 21','Pedra 22','Pedra 23','INDE 22','INDE 23','Cg','Cf','Ct'
                   ,'Avaliador1','Rec Av1','Avaliador2','Rec Av2','Avaliador3','Rec Av3','Avaliador4'
                   ,'Rec Av4','Rec Psicologia','Indicado','Atingiu PV','Destaque IEG','Destaque IDA'
                   ,'Destaque IPV','Destaque IPV.1']

base_2023.drop(columns=colunas_remover2023, inplace=True)

In [191]:
#remover colunas dos dataframes 2024 
colunas_remover2024 = ['Data de Nasc','Pedra 20', 'Pedra 21','Pedra 22','Pedra 23','INDE 22','INDE 23','Cg','Cf','Ct'
                   ,'Avaliador1','Rec Av1','Avaliador2','Rec Av2','Avaliador3','Avaliador4','Avaliador5'
                   ,'Avaliador6', 'Rec Psicologia','Indicado','Atingiu PV','Destaque IEG','Destaque IDA'
                   ,'Destaque IPV','Escola','Ativo/ Inativo','Ativo/ Inativo.1']

base_2024.drop(columns=colunas_remover2024, inplace=True)

## Ajustes nas colunas

In [192]:
# Vamos padronizar os nomes das colunas
base_2022 = normalizar_colunas(base_2022)
base_2023 = normalizar_colunas(base_2023)
base_2024 = normalizar_colunas(base_2024)

<div class="alert alert-block alert-info">
⚠️ **IMPORTANTE VALIDAR  ALTERACAO QUE VITOR REALIZOU**
</div>

In [193]:
#Não identifiquei mas por "dedução fiz um de para na coluna FASE na base 2024"
""" 
Alterado de:
array(['ALFA', '1A', '1B', '1C', '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R', '2A', '2B', '2C', '2D', '2G', '2H'
        , '2I', '2K', '2L', '2M', '2N', '2P', '2R', '2U', '3A', '3B', '3C', '3D', '3F', '3G', '3H', '3I', '3K', '3L', '3M', '3N', '3P'
        , '3R', '3U', '4A', '4B', '4C', '4F', '4H', '4L', '4M', '4N', '4R', '5A', '5B', '5C', '5D', '5F', '5G', '5L', '5M', '5N', '6A'
        , '6L', '7A', '7E', '8A', '8B', '8D', '8E', '8F', 9], dtype=object) 

Para: array(['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8', 'FASE 9'], dtype=object) 
"""

base_2024 = base_2024.rename(columns={"fase": "fase_original"})
base_2024["fase"] = base_2024["fase_original"].map(normalizar_fase)

In [194]:
base_2024['fase'].value_counts().sort_index()


fase
ALFA      196
FASE 1    185
FASE 2    185
FASE 3    211
FASE 4    115
FASE 5    100
FASE 6     25
FASE 7     37
FASE 8     64
FASE 9     38
Name: count, dtype: int64

![Quantidade total alunos fases](AlunosFases_2024.jpg)


Fonte da imagem anterior com totais de alunos por fase de 2024, retirado do documento:

https://passosmagicos.org.br/wp-content/uploads/2025/05/relatorio_de_atividades_2024_compressed.pdf


### Temos uma diferença de 1 registro para a fase 4 (nosso de para chegou em 115 e no documento informa 114)

### Temos uma diferença de 1 registro para a fase 6 (nosso de para chegou em 25 e no documento informa 24)

<div class="alert alert-block alert-info">
⚠️ **PODEMOS TOMAR COMO CORRETO ESTE DE PARA FEITO???**
</div>

In [195]:
#Guardando informações origiais de fase_ideal antes de fazer o map

base_2022 = base_2022.rename(columns={"fase_ideal": "fase_ideal_original"})
base_2022["fase_ideal"] = base_2022["fase_ideal_original"].map(remover_texto_parenteses)

base_2023 = base_2023.rename(columns={"fase_ideal": "fase_ideal_original"})
base_2023["fase_ideal"] = base_2023["fase_ideal_original"].map(remover_texto_parenteses)

base_2024 = base_2024.rename(columns={"fase_ideal": "fase_ideal_original"})
base_2024["fase_ideal"] = base_2024["fase_ideal_original"].map(remover_texto_parenteses)

In [196]:
#Unificando todas as bases em um unico dataframe

base_anos = pd.concat(
    [base_2022, base_2023, base_2024],
    axis=0,        # empilha linhas
    ignore_index=True,
    sort=False     # mantém todas as colunas
)

In [197]:
#Colunas finais:
base_anos.columns

Index(['ra', 'fase', 'turma', 'nome_anonimizado', 'idade', 'genero',
       'ano_ingresso', 'instituicao_de_ensino', 'pedra', 'inde', 'no_av',
       'iaa', 'ieg', 'ips', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian',
       'fase_ideal_original', 'defasagem', 'ano_aba', 'fase_ideal', 'ipp',
       'fase_original'],
      dtype='object')

### Ajustando dados de Pedra (pois continha Ágata e Agata) além de padronizar a coluna genero

In [198]:
base_anos['pedra'].unique()

array(['Quartzo', 'Ametista', 'Ágata', 'Topázio', 'Agata', nan, 'INCLUIR'],
      dtype=object)

In [199]:
#Ajustando o nome da Pedra para mantermos um padrão definido no documento da Passos Magicos
base_anos['pedra'] = base_anos['pedra'].replace('Agata', 'Ágata')

In [200]:
base_anos["genero"] = base_anos["genero"].apply(genero_norm)

### Salvando o arquivo completo sem nenhuma exclusão de linhas

In [ ]:
#base_anos.to_excel('base_anos.xlsx', index=False)
#print("Arquivo 'meus_dados.xlsx' criado com sucesso!")

Arquivo 'meus_dados.xlsx' criado com sucesso!


### Vamos excluir os que não possuem nenhum tipo de informação em Pedra?

In [202]:
#Remoção de linhas onde não temos identificação da informação de PEDRA (optamos por excluir estas linhas para analise das informações)
base_filtrado = base_anos[
    base_anos['pedra'].isna() |
    (base_anos['pedra'].astype(str).str.strip() == '') |
    (base_anos['pedra'].astype(str).str.strip() == 'INCLUIR')
]

In [203]:
#Quantidade de linhas que vamos remover por não ter informações relevantes na coluna PEDRA
base_filtrado.shape

(185, 26)

In [204]:
base_anos_limpo = base_anos.drop(index=base_filtrado.index)

### Salvando o arquivo com exclusão de 185 linhas (coluna PEDRA nula ou em branco ou INCLUIR)

In [205]:
base_anos_limpo.to_excel('base_anos_limpo.xlsx', index=False)
print("Arquivo 'meus_dados.xlsx' criado com sucesso!")

Arquivo 'meus_dados.xlsx' criado com sucesso!


Após a exportação para arquivos EXCEL realizamos a publicação no GIT para facilitar a utilização de um caminho online dos arquivos

# IMPORTANDO DATAFRAME "LIMPO"

In [ ]:
#EXEMPLO PARA IMPORTAR AS BASES EXPORTADAS:
url = "https://raw.githubusercontent.com/vbomura/tc5/main/Codigos/base_anos_limpo.xlsx"
dfBaseAnosLimpos = pd.read_excel(url)